# Streamflow Getter 

**Project lead and repository author:** Professor James Halgren  
**Research collaborator:** Gonzalo Forero Buitrago

Select USGS/NWM/GEOGLOWS reaches, retrieves observed and modeled discharge.

## A | Local installation and Jupyter kernel (terminal only)

These steps reproduce the working **macOS + `uv` + VS Code** setup. Perform them in a terminal, not as notebook cells. 

### A1. Required files and tools

Install **Git**, **`uv`**, **VS Code**, and its **Python** and **Jupyter** extensions. 

The example reads `~/Documents/streamflow_getter_guide_install/streamflow_getter_config.json`. It must contain valid `routelink`, `hydrofabric` and `workspace` paths. You can point to another configuration file with `STREAMFLOW_GETTER_CONFIG`. Download/provision those scientific reference datasets separately; never commit local paths, API credentials, or research outputs.

### A2. Create or synchronize `.venv` and install dependencies

`uv sync` **creates the repository's `.venv` if it does not exist**, installs the local package and its optional dependencies, and uses the committed lock file. It must include `nbformat` (needed by Plotly in notebooks). The `notebook` extra was updated during development to include it; **do not run `uv add` every session**.

```bash
cd ~/Documents/streamflow_getter || exit 1
uv sync --locked --extra operational --extra spatial --extra parquet --extra notebook

# Check the actual package environment and Plotly's notebook dependency:
uv run --locked --extra notebook python -c   'import sys, nbformat, plotly, ipykernel; print(sys.executable); print("nbformat:", nbformat.__version__)'
```

If you are preparing a *different* checkout and `nbformat` is genuinely absent from its `notebook` extra, update **that checkout's** dependency declaration and lock file once, review those changes, then synchronize; do not rely on a temporary `pip install` fix.

### A3. Register the Jupyter kernel

```bash
cd ~/Documents/streamflow_getter || exit 1
uv run --locked --extra operational --extra spatial --extra parquet --extra notebook   python -m ipykernel install --user   --name streamflow-getter-package   --display-name "Python (Streamflow Getter package)"

uv run --locked --extra notebook jupyter kernelspec list
```

The kernel is registered under `~/Library/Jupyter/kernels/streamflow-getter-package`. In VS Code open this notebook and choose **Select Kernel → Python (Streamflow Getter package)**. If the name is hidden, choose **Select Another Kernel → Python Environments** and select `~/Documents/streamflow_getter/.venv/bin/python`; then run **Developer: Reload Window** and restart the notebook kernel. Registering a kernel and selecting it in VS Code are distinct steps.

### A4. Quality checks and notebook launch

```bash
cd ~/Documents/streamflow_getter || exit 1
uvx ruff check examples/streamflow_getter_package_demo_v3.ipynb
uvx ruff format --check examples/streamflow_getter_package_demo_v3.ipynb
uv run --locked --extra operational --extra spatial --extra parquet --extra notebook   python -m pytest -q
code examples/streamflow_getter_package_demo_v3.ipynb
```

**Daily use:** after dependencies and the kernel are ready, select the kernel, restart it, and run B → C → D1 → D2 → E1–E6. You do **not** need to reinstall or re-register on every session.


## B | Load the package and record provenance

Import the **installed `streamflow_getter` package** directly. Verify the active `.venv`, local study configuration, RouteLink/Hydrofabric paths, and `nbformat`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

import nbformat
import pandas as pd
import plotly.io as pio
from IPython.display import display
from ipywidgets import VBox

import streamflow_getter
from streamflow_getter import ComparisonResult, StreamflowClient
from streamflow_getter.interactive_map import StreamflowMap
from streamflow_getter.roadmap_diagnostics import (
    benchmark_access,
    benchmark_cache,
    benchmark_parquet,
    diagnose_click_selection,
    diagnose_reaches,
    summarize_exact_utc_overlap,
)

CONFIG_FILE = Path(
    os.environ.get(
        "STREAMFLOW_GETTER_CONFIG",
        Path.home()
        / "Documents/streamflow_getter_guide_install/streamflow_getter_config.json",
    )
).expanduser()
if not CONFIG_FILE.is_file():
    raise FileNotFoundError(f"Missing study configuration: {CONFIG_FILE} (see A1)")
settings = json.loads(CONFIG_FILE.read_text(encoding="utf-8"))
if not isinstance(settings, dict):
    raise ValueError("The configuration must contain a JSON object")
for name in ("routelink", "hydrofabric"):
    if name not in settings or not Path(settings[name]).expanduser().exists():
        raise FileNotFoundError(f"Missing {name} file/directory in configuration")

# Presentation only: no scientific computations are defined in this notebook.
pio.renderers.default = "vscode"
checkout = Path(settings.get("streamflow_repo") or Path.cwd()).expanduser()
revision = (
    subprocess.run(
        ["git", "-C", str(checkout), "rev-parse", "--short", "HEAD"],
        capture_output=True,
        text=True,
        check=False,
    ).stdout.strip()
    or None
)
if getattr(StreamflowMap, "INTERFACE_VERSION", 0) < 23:
    raise RuntimeError("Outdated interactive-map backend; synchronize the package")

print("Python:", sys.executable)
print("Package:", streamflow_getter.__file__)
print("Backend Git revision:", revision or "unavailable")
print("Plotly notebook support: nbformat", nbformat.__version__)
print("RouteLink:", settings["routelink"])
print("Hydrofabric:", settings["hydrofabric"])

## C | Study configuration and saving options

**Example:** USGS `08165500` near Hunt, Texas; reference time **2025-07-04 06:00 UTC**. Discharge is expressed in **m³/s**. At this date the operational NWM data are **version 3.0**; the locally available **RouteLink is version 3.1**.

**Two controls are deliberately separate:**

| Setting | What it controls | With `True` |
|---|---|---|
| `SAVE_CSV` | Whether the backend also requests CSV output in its **normal station-save workflow**; it does not start saving | CSV output is requested when a station is saved; check the reported files for the complete output set |
| `SAVE_STATION_AFTER_QUERY` | Whether to save the C2 study immediately after retrieval | Calls the existing `study.save_record(record)` once |



In [ ]:
USGS_GAGE = "08165500"  # 08165300, 08165500
REFERENCE_UTC = "2025-07-04T06:00:00Z"
NWM_VERSION_CHOICE = "auto"  # Select the model data release for this date.
CROSS_VERSION_POLICY = (
    "exploratory"  # Exploratory candidate; 'strict' rejects mismatched mappings.
)
HOURS_BEFORE, HOURS_AFTER = 3, 6
SEARCH_RADIUS_M = 3000
MAX_WORKERS = 5
SAVE_CSV = True  # Format option for manual saving; not a save trigger.
SAVE_STATION_AFTER_QUERY = True  # True: save the reference station once in C2.
SAMPLE_LEAD_HOUR = 1  # E1/E3: valid time = forecast initialization + 1 hour.

### C1 | Initialize one study through `StreamflowClient`

 A message such as **`NWM 3.0 versus RouteLink 3.1: INCOMPATIBLE`** means the **release versions are different**. 


In [ ]:
client = StreamflowClient(
    settings,
    hours_before=HOURS_BEFORE,
    hours_after=HOURS_AFTER,
    nwm_version=NWM_VERSION_CHOICE,
    cross_version_policy=CROSS_VERSION_POLICY,
    search_radius_m=SEARCH_RADIUS_M,
    max_workers=MAX_WORKERS,
    save_csv=SAVE_CSV,
    auto_save=False,
    backend_revision=revision,
)
study = client.study(REFERENCE_UTC)
print(study.network_message)
print("UTC window:", study.start, "→", study.end)
print(f"Backend initialization: {study.initialization_seconds:.2f} s")
if not study.crosswalk_compatible:
    print(
        "Network note: different NWM/RouteLink releases, not different observation dates."
    )
    print("RouteLink selects an exploratory candidate; Hydrofabric supplies geometry.")
    print(
        "The candidate's validity for this model release is NOT independently verified."
    )

### C2 | Retrieve and inspect the reference station

The **backend** joins the USGS station to a candidate NWM `feature_id` using **RouteLink**, looks up the reach geometry using **Hydrofabric**, selects GEOGLOWS independently, retrieves available discharge, and aligns observed/model values on **exact UTC timestamps (no interpolation)**.

**How to interpret the warnings:**

- `RouteLink verified for NWM version: False` — a candidate reach *was selected*, but the available **3.1 crosswalk has not been validated against the 3.0 model network** 
- `NWM Hydrofabric network compatibility is not verified` — the available river geometry dataset has **not** independently been established to match the NWM data release. 
- `Hydrologic equivalence ... not verified` — nearby/overlapping NWM and GEOGLOWS river lines **do not prove the same upstream drainage area or channel topology**. Additional validation would be needed before interpreting model differences as skill.


In [ ]:
comparison = ComparisonResult(study.compare_station(USGS_GAGE))
record = comparison.raw  # Original backend record for maps and diagnostics.
meta = comparison.metadata

print("Selected identifiers:", comparison.ids)
print("NWM candidate selected via:", meta["nwm"]["selection_method"])
print(
    "USGS→NWM mapping verified for this model release:",
    meta["nwm"]["gage_crosswalk_verified_for_version"],
)
print("Drainage-area context:", meta["drainage_area_comparison"]["message"])
print("Available discharge samples:", comparison.source_rows)
print(
    "Total request wall time (s):",
    round(meta["timings_seconds"]["total_request_wall"], 2),
)
print(
    "Concurrent provider durations (s):",
    {
        name: round(seconds, 2)
        for name, seconds in meta["timings_seconds"]["per_source"].items()
    },
)
if comparison.warnings:
    print("Scientific limitations (not retrieval failures):")
    for warning in comparison.warnings:
        print("  -", warning)

if SAVE_STATION_AFTER_QUERY:
    print("Station-save result:", study.save_record(record))
else:
    print("No permanent station file saved by C2. Use D1 Save if needed.")

## D1 | Fixed USGS station: map and hydrograph

The panel remains tied to station `08165500`. It shows the **actual selected** USGS/NWM/GEOGLOWS series and geometry/area context

In [ ]:
previous_view = globals().get("river_view")
if previous_view is not None:
    previous_view.close()
river_view = StreamflowMap(study, record)
display(
    VBox(
        [
            river_view.station_status,
            river_view.station_map,
            river_view.station_hydrograph,
            river_view.station_save,
        ]
    )
)

## D2 | Independent interactive river explorer

Enter a date (`2025-07-04`) or explicit hour (`2025-07-04T06:00:00Z`), then **click a river**. The backend selects the nearby NWM reach using Hydrofabric and the GEOGLOWS reach on **its independent network**, then updates the D2 hydrograph. USGS is included only when a mapped station is sufficiently close to the **current click**. Each new selection replaces the previous D2 result, without changing D1. An unavailable/invalid remote geometry should produce a warning without blocking the next click.


In [ ]:
if not hasattr(river_view, "reference_input"):
    raise RuntimeError(
        "The current kernel lacks the date-selectable map interface. "
        "Update the editable package and restart the kernel."
    )
river_view.reference_input.value = REFERENCE_UTC
print("Explorer ready. Change UTC date/time below, then click a river.")
print("The hydrograph below the map will update for each click.")

In [ ]:
display(
    VBox(
        [
            river_view.reference_input,
            river_view.status,
            river_view.map,
            river_view.hydrograph,
            river_view.click_save,
        ]
    )
)
# Latest selected point: river_view.result | ongoing query: river_view.task

### How reach identity and distance are handled

1. **D1 (station):** RouteLink performs a **gage ID → NWM feature ID** lookup; Hydrofabric provides a river line for that feature.
2. **D2 (free click):** a **nearest-river geographic search** in the Hydrofabric is used instead. An arbitrary point does not first have a USGS identifier. GEOGLOWS is queried separately; its IDs are **not** NWM IDs.
3. **Distance:** USGS coordinates are EPSG:4326 and source Hydrofabric coordinates EPSG:4269. The backend projects relevant lines/points to EPSG:5070 to measure **planar meters**. A 0-m minimum line-to-line separation may mean lines overlap or intersect; it does **not** prove identical network topology or upstream basins.

 Discharge is **m³/s**, all valid timestamps are **UTC**, and no samples are interpolated to make the time series appear synchronized.


## E | Six reproducibility and interpretation experiments

**E1/E3 make new remote requests** for a single NWM forecast hour; their timing/memory numbers are illustrative, not a rigorous performance benchmark. 
**E2** checks network selection and geometry 
**E4** checks data preservation through file I/O
**E5** counts exact UTC overlap, and 
**E6** exercises export/verification/offline reload.

### E1 | Kerchunk vs native NetCDF: same NWM value, two readers

**`+1 h` = forecast lead time.** If a forecast was initialized at **2025-07-04 06:00 UTC**, lead **+1 h** corresponds to **07:00 UTC**. This is the **same reach, model product, initialization and valid time** for both readers; only the retrieval method changes.

- **Kerchunk:** reads selected data through references into remote NetCDF content.
- **Native:** opens the remote NetCDF through the native access route.
- **RSS (resident set size):** approximate RAM occupied by the current Python process. **MiB** = mebibyte = 1,048,576 bytes. **`extra RSS`** is an observed change in process memory, **not download size or file size**.
- **Flow matches:** both access routes returned numerically equivalent discharge. The tolerance (`1e-6`) is in m³/s.

Wall times and RSS vary with network state, cached metadata, memory reuse and measurement timing. A single pair of requests cannot establish which method is universally faster or smaller.


In [ ]:
feature_id = meta["ids"]["nwm"]
if feature_id is None:
    print("No NWM candidate selected; E1 skipped (USGS/GEOGLOWS remain available).")
else:
    access = benchmark_access(
        study.reference.isoformat(),
        feature_id,
        lead_hour=SAMPLE_LEAD_HOUR,
        cache_dir=study.workspace / "reference_cache",
    )
    valid_utc = (study.reference + pd.Timedelta(hours=SAMPLE_LEAD_HOUR)).isoformat()
    print(
        f"NWM {study.version} | reach {feature_id} | lead +{SAMPLE_LEAD_HOUR} h | valid {valid_utc}"
    )
    for method in ("kerchunk", "native"):
        item = access[method]
        if item["ok"]:
            print(
                f"{method:8} | {item['streamflow_m3s']:.5f} m³/s | "
                f"{item['seconds']:.2f} s | RSS change ~{item['rss_increment_mb']:.1f} MiB"
            )
        else:
            print(f"{method:8} | unavailable: {item['error']}")
    print("Both readers succeeded:", access["comparable"])
    print("Flows agree (±1e-6 m³/s):", access["equal_within_tolerance"])
    print("Absolute difference (m³/s):", access["absolute_difference_m3s"])

### E2 | RouteLink → Hydrofabric → map: what is actually checked?

**NWM 3.0** identifies the **discharge model release** for the requested time. **RouteLink 3.1** identifies the **station-to-reach mapping file release** installed locally. The map can draw a line with an ID selected from RouteLink and found in Hydrofabric, while the **cross-version mapping remains unverified**. Thus `compatible: False` is **not the same as “no map” or “no data.”

**Why can two distances appear contradictory?** `NWM river → GEOGLOWS river: 0.00 m` measures the **minimum separation of the two plotted river geometries**; 0 m can result from an intersection/overlap. `Independent GEOGLOWS service distance: ~106.8 m` describes the GEOGLOWS selection's **distance from its query location to the selected GEOGLOWS channel**. These are **different pairs of geometric objects** and should not be compared as if they were the same distance. Close lines do not prove the same river network or drainage basin.

In [ ]:
spatial = diagnose_reaches(study, record, USGS_GAGE, SEARCH_RADIUS_M)
lon, lat = spatial["station_epsg4326_lonlat"]
x, y = spatial["station_epsg5070_xy_m"]
print(f"USGS {USGS_GAGE}: (lon, lat) = ({lon:.6f}, {lat:.6f}) EPSG:4326")
print(f"Projected station: ({x:,.1f}, {y:,.1f}) m EPSG:5070")
print("RouteLink gage→feature_id:", spatial["feature_id"])
print("D1 mapped NWM feature_id:", spatial["map_nwm_id"])
print("Same ID selected and displayed:", spatial["routelink_matches_map_id"])
if spatial["routelink_matches_map_id"] is False:
    raise RuntimeError("The displayed NWM ID disagrees with the RouteLink lookup")
print("NWM geometry matches found:", spatial["nwm_reach_count"])
print(
    "Hydrofabric vs displayed NWM geometry (m):",
    spatial["hydrofabric_vs_map_geometry_m"],
)
print(
    f"Release compatibility: NWM {spatial['nwm_version']} / "
    f"RouteLink {spatial['routelink_network_version']} → "
    f"verified for this release: {spatial['crosswalk_compatible']}"
)
print("D1 GEOGLOWS river ID:", spatial["map_geoglows_id"])
for label, key in (
    ("Station → NWM line", "usgs_to_nwm_m"),
    ("Station → GEOGLOWS line", "usgs_to_geoglows_m"),
    ("NWM line → GEOGLOWS line", "nwm_to_geoglows_m"),
):
    distance = spatial[key]
    print(
        f"{label}: {distance:.2f} m"
        if distance is not None
        else f"{label}: unavailable"
    )
print(
    "GEOGLOWS lookup-point → chosen river (m):", spatial["geoglows_service_distance_m"]
)
print("Nearby Hydrofabric candidates (NOT substitutions for RouteLink):")
for candidate in spatial["nearby"]:
    print(f"  {candidate['feature_id']}: {candidate['distance_m']:.1f} m")
print("Drainage-area context:", spatial["area"])
if spatial["geometry_warning"]:
    print("Optional geometry warning:", spatial["geometry_warning"])
print(f"Spatial diagnostic: {spatial['seconds']:.2f} s")

if river_view.result is not None:
    clicked = diagnose_click_selection(river_view._explorer_study, river_view.result)
    print("D2 most recent click | map/backend nearest-reach consistency:")
    for label, key in (
        ("Displayed NWM ID", "map_nwm_id"),
        ("Backend nearest NWM ID", "backend_nearest_nwm_id"),
        ("Identifiers agree", "matches_backend_nearest"),
        ("USGS mapped at click", "map_usgs_id"),
        ("USGS/RouteLink agreement", "matches_routelink_when_usgs"),
        ("Independent GEOGLOWS ID", "map_geoglows_id"),
    ):
        print(f"  {label}: {clicked[key]}")
else:
    print(
        "D2 has not been clicked; click a river and rerun E2 to check that selection."
    )

### E3 | Two repeated reads: cache consistency, not two forecasts

The **operational coordinator** is the package component that **organizes a targeted NWM request**: it chooses **Kerchunk references** when available, reuses cached reference metadata, and can fall back to **native remote NetCDF** for specific recoverable access failures. It does **not** generate or change a forecast.

- **Request 1:** read **one NWM reach and one forecast hour**.
- **Request 2:** repeat the **exact same reach/hour/product** using the **same reference cache**. It is **not a different forecast**, a new initialization, or another USGS station.
- **`kerchunk` on both:** both calls were served by that access method. **`native_remote`** would mean the approved native fallback was used.
- **`Same discharge: True`:** repeated retrieval produced the same numerical value.
- **`Second minus first: -0.38 s`:** the second read happened to be 0.38 seconds faster. This demonstrates possible reuse/caching, **not a statistically meaningful speed improvement**; network, OS caches and other factors also change.



In [ ]:
timing = meta["timings_seconds"]
print(f"C2 multi-provider wall time: {timing['total_request_wall']:.2f} s")
print("C2 concurrent provider durations (not additive):")
for name, seconds in timing["per_source"].items():
    print(f"  {name:22} {seconds:.2f} s")

if feature_id is None:
    print("No NWM reach; skip the repeated one-hour request.")
else:
    cache_run = benchmark_cache(
        study.reference.isoformat(),
        feature_id,
        lead_hour=SAMPLE_LEAD_HOUR,
        cache_dir=study.workspace / "reference_cache",
    )
    print("Repeated reads of ONE identical NWM reach, product and valid hour:")
    for item in cache_run["requests"]:
        label = "first read" if item["iteration"] == 1 else "repeat read"
        if item["ok"]:
            print(
                f"  {label}: {item['method']} | {item['seconds']:.2f} s | {item['streamflow_m3s']:.5f} m³/s"
            )
        else:
            print(f"  {label}: unavailable — {item['error']}")
    if len(cache_run["requests"]) == 2 and all(x["ok"] for x in cache_run["requests"]):
        first, second = cache_run["requests"]
        print(f"Second minus first: {second['seconds'] - first['seconds']:+.2f} s")
    print("Numerical discharge consistent:", cache_run["equal_within_tolerance"])
    print("Two reads demonstrate consistency; they are not a formal speed benchmark.")

### E4 | Parquet round-trip: does the file preserve the data?

A **round-trip** means **write a pandas DataFrame to Parquet → read that Parquet back → compare the recovered DataFrame with the original**. The package's `benchmark_parquet()` checks values, column names, UTC timestamps and **missingness**. `Exact ... round-trip: True` means this test recovered the table without differences; it does **not** evaluate model accuracy.

`Non-null discharge samples by source` counts timestamps with an **actual discharge value** for each source (`NaN`/missing values are excluded; **0.0 m³/s is a valid value and is counted**). Counts can differ because providers have different availability, time steps and missing observations. The aligned table is a **union of UTC timestamps**, not a promise that all sources have data at every row. Local read/write times do not include remote downloads. The test uses temporary storage, not a permanent saved study.


In [ ]:
parquet = benchmark_parquet(record, USGS_GAGE)
print("Original DataFrame == Parquet reload (exact):", parquet["equal"])
print("Unique UTC timestamps:", parquet["rows"])
print("Columns (valid_time + discharge series):", parquet["columns"])
print("Actual available values per source (NaN excluded; zero included):")
for column, count in parquet["source_columns"].items():
    print(f"  {column.removesuffix('_streamflow_m3s'):24} {count}")
print(f"Parquet file size: {parquet['bytes']:,} bytes")
print(
    f"Temporary write/read: {parquet['write_seconds']:.3f} / {parquet['read_seconds']:.3f} s"
)
print("Permanent file created:", parquet["saved_permanently"])

### E5 | Exact UTC overlap is not forecast skill

This experiment calls `summarize_exact_utc_overlap()` to report how many **USGS observations** and **model values** share the **same UTC timestamp**. It performs **no interpolation**, does not invent missing observations, and does not calculate skill metrics such as bias, RMSE or NSE.

- **`NWM/RouteLink association verified: False`** — the 3.1 gage-to-reach index used for exploratory selection has **not been version-validated** for the older 3.0 NWM data. Some values may still have been retrieved and plotted; the warning is about the **evidence supporting the mapping**.
- **`Forecast skill calculated: False`** — this cell only counts exact-time pairs, **not** how accurate NWM/GEOGLOWS predictions were compared with USGS. That requires an appropriate verified location, product, forecast/reference-time definition and skill method.
- **`No timestamp interpolation or hydrologic equivalence is implied`** — no data were shifted/filled to make time points coincide; nor does time overlap prove the model reaches represent the same watershed.

In [ ]:
overlap = summarize_exact_utc_overlap(record)
print("Time basis:", overlap["time_basis"])
print("USGS available samples:", overlap["usgs_samples"])
print("Model source                  available     paired with USGS at identical UTC")
for name, counts in overlap["model_sources"].items():
    print(
        f"{name:28} {counts['model_samples']:>7}"
        f" {counts['paired_with_usgs_at_exact_utc']:>12}"
    )
print(
    "NWM/RouteLink model-release association verified:",
    overlap["crosswalk_verified_for_nwm"],
)
print("Forecast skill calculated:", overlap["skill_evaluated"])
print(
    "Interpretation: exact timestamps only; no interpolation or catchment-equivalence claim."
)

### E6 | Package export → SHA-256 integrity → offline reload (corrected)

This test exports **the real C2 study** through `ComparisonResult.export()` as **Parquet plus versioned JSON manifest**, then calls `ComparisonResult.load()` **offline**. The package validates **SHA-256 and file size** before decoding the data. The notebook checks that the recovered table, UTC timestamps, missing values, scientific identifiers, warnings, and **serialized GeoJSON** match their expected representations. Temporary files are automatically deleted afterward.

**Why the v2 assertion was unreliable:** it compared **raw in-memory geometries** with **serialized/reloaded GeoJSON**. Those representations are not guaranteed to compare equal in Python (for example, tuples and JSON arrays become different Python types). The failed assertion alone does **not** establish that any river line changed. V3 compares **`recovered.geometries == manifest["geometries"]`**, verifying the actual exported GeoJSON without reimplementing the package's geometry serializer.

**Important distinction:** E6 proves integrity and reproducible **serialization**, not independent topological, spatial or hydrologic equivalence between NWM and GEOGLOWS.


In [ ]:
with TemporaryDirectory(prefix="streamflow-getter-demo-") as temporary_directory:
    data_path, manifest_path = comparison.export(
        Path(temporary_directory) / "comparison.parquet"
    )
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    integrity = manifest["data_integrity"]
    assert manifest["schema"] == "streamflow_getter.comparison_export.v2"
    assert integrity["algorithm"] == "sha256"
    assert integrity["size_bytes"] == data_path.stat().st_size

    # load() validates SHA-256; no NOAA, USGS or GEOGLOWS query is made here.
    recovered = ComparisonResult.load(data_path)
    pd.testing.assert_frame_equal(recovered.table, comparison.table, check_exact=True)
    assert recovered.ids == comparison.ids
    assert recovered.metadata == manifest["metadata"]
    assert recovered.geometries == manifest["geometries"]  # Saved canonical GeoJSON.
    assert recovered.warnings == comparison.warnings

    print("Export schema:", manifest["schema"])
    print("Rows and columns:", recovered.table.shape)
    print("USGS / NWM / GEOGLOWS IDs:", recovered.ids)
    print("SHA-256 checked by package:", integrity["sha256"])
    print("Data bytes:", integrity["size_bytes"])
    print("PASS: UTC, discharge, gaps, IDs, serialized GeoJSON and warnings")
print("PASS: Offline reload; temporary files removed")

### Reproducibility notes and future presentation layer

**Original author / project lead:** Professor James Halgren. **Research assistant / collaborator:** Gonzalo Forero Buitrago. 

**Architecture:** `StreamflowClient` creates the study; `ComparisonResult` exposes its data and supports export/reload. The package owns the USGS, NWM and GEOGLOWS service access; NWM request coordination; RouteLink/Hydrofabric/GEOGLOWS reach resolution; CRS and distance calculations; exact UTC alignment; caching; diagnostics and integrity checks. The notebook **only configures, calls, interprets and displays the package**. D1/D2 are demonstration views; a future standalone web visualizer should use the same API and saved-result contract, not copy the backend logic.